# 05 — Thermo output to a file

Every other notebook so far pulls thermo data straight out of the live
`lmp` object. That is not how most LAMMPS workflows work on a real cluster:
you submit a script, it runs to completion, and you analyze the numbers it
left behind on disk. This notebook writes time, temperature and energy to a
file with `fix ave/time` — the same technique you would use with native
LAMMPS — and a separate cell reads that file back and plots it, with no
live connection to the simulation at all.

In [ ]:
%pip install lammps-js matplotlib

## Run the script

`fix ave/time` samples equal-style variables on a schedule and writes them to
a file as plain columns. Here it samples every 10 steps
(`Nevery=10 Nrepeat=1 Nfreq=10`, i.e. one row per 10-step window) and writes
to `thermo.txt`. The output is silenced (`output=None`) to make the point:
nothing is collected from the live simulation — everything goes to the file:

In [ ]:
from lammps import lammps

lmp = await lammps(output=None)
lmp.commands_string("""
units         lj
atom_style    atomic
lattice       fcc 0.8442
region        box block 0 4 0 4 0 4
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 3.0 87287
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
thermo        200

variable      t equal time
variable      T equal temp
variable      pe equal pe
variable      etot equal etotal
fix           thermo_out all ave/time 10 1 10 v_t v_T v_pe v_etot file thermo.txt

run           4000
""")
lmp.close()
print("done — thermo.txt has the time series")

## Read the file back and plot

The simulation is closed — all that's left is `thermo.txt`. LAMMPS wrote it
straight to the notebook filesystem (check the file browser on the left), so
plain `numpy.loadtxt` reads it like any file on disk. The `#` header lines
`fix ave/time` writes are skipped by `loadtxt` automatically:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

step, t, temp, pe, etot = np.loadtxt("thermo.txt", unpack=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(t, temp)
ax1.set(xlabel="time", ylabel="temperature", title="Melting: T relaxes")

ax2.plot(t, pe, label="potential")
ax2.plot(t, etot, label="total")
ax2.set(xlabel="time", ylabel="energy / atom", title="NVE: total energy is conserved")
ax2.legend()

fig.tight_layout()
plt.show()

Back to [the notebook index](../index.ipynb).